# OASIS Cross-Sectional Dataset — Preprocessing Pipeline

**Goal:** Transform the raw OASIS data into a clean, numeric, model-ready format for Alzheimer's Disease classification (Demented vs. Nondemented).

**Key design decisions (carried over from EDA):**
- `Group` does not exist in the raw file — it is derived from `CDR`. Subjects with missing `CDR` were never clinically assessed for dementia and are **dropped** from the modeling set rather than imputed (imputing a diagnosis would fabricate ground truth).
- `Delay` is missing for ~95% of subjects and carries no useful signal — dropped.
- `ID` is a unique identifier with no predictive value — dropped.
- `eTIV` and `ASF` are highly correlated (near-redundant) but both are kept here; redundancy removal is a modeling-stage decision (e.g. via feature selection or regularization), not a preprocessing one.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42


## Step 0: Load Dataset

In [2]:
DATA_PATH = "oasis_cross-sectional.csv"  # update path if needed

df = pd.read_csv(DATA_PATH)
print(f"Initial dataset shape: {df.shape}")
df.head()


Initial dataset shape: (436, 12)


,ID,M/F,Hand,Age,Educ,SES,MMSE,CDR,eTIV,nWBV,ASF,Delay
0,OAS1_0001_MR1,F,R,74,2.0,3.0,29.0,0.0,1344,0.743,1.306,NaN
1,OAS1_0002_MR1,F,R,55,4.0,1.0,29.0,0.0,1147,0.810,1.531,NaN
2,OAS1_0003_MR1,F,R,73,4.0,3.0,27.0,0.5,1454,0.708,1.207,NaN
3,OAS1_0004_MR1,M,R,28,NaN,NaN,NaN,NaN,1588,0.803,1.105,NaN
4,OAS1_0005_MR1,M,R,18,NaN,NaN,NaN,NaN,1737,0.848,1.010,NaN


## Step 4 (early): Create Target Variable from CDR

We create the target before dropping rows, since the target itself determines which rows are valid for modeling (subjects without a CDR score have no ground-truth label).

In [3]:
def cdr_to_group(cdr):
    if pd.isna(cdr):
        return "Unknown"
    elif cdr == 0:
        return "Nondemented"
    else:
        return "Demented"

df["Group"] = df["CDR"].apply(cdr_to_group)

print("Group distribution (including Unknown):")
print(df["Group"].value_counts())
print(f"\nShape after adding Group column: {df.shape}")


Group distribution (including Unknown):
Group
Unknown        201
Nondemented    135
Demented       100
Name: count, dtype: int64

Shape after adding Group column: (436, 13)


In [4]:
# Drop subjects with no clinical diagnosis (Unknown) -- cannot be used for supervised classification
df = df[df["Group"] != "Unknown"].reset_index(drop=True)

print(f"Shape after removing 'Unknown' diagnosis subjects: {df.shape}")
print(df["Group"].value_counts())


Shape after removing 'Unknown' diagnosis subjects: (235, 13)
Group
Nondemented    135
Demented       100
Name: count, dtype: int64


## Step 1: Remove Unnecessary Columns

In [5]:
cols_to_drop = ["ID", "Delay", "CDR"]
# ID: unique identifier, no predictive value
# Delay: ~95% missing, no usable signal
# CDR: this is the variable Group was derived FROM -- keeping it would leak the label directly into the features

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Dropped columns: {cols_to_drop}")
print(f"Shape after dropping unnecessary columns: {df.shape}")
df.head()


Dropped columns: ['ID', 'Delay', 'CDR']
Shape after dropping unnecessary columns: (235, 10)


,M/F,Hand,Age,Educ,SES,MMSE,eTIV,nWBV,ASF,Group
0,F,R,74,2.0,3.0,29.0,1344,0.743,1.306,Nondemented
1,F,R,55,4.0,1.0,29.0,1147,0.810,1.531,Nondemented
2,F,R,73,4.0,3.0,27.0,1454,0.708,1.207,Demented
3,M,R,74,5.0,2.0,30.0,1636,0.689,1.073,Nondemented
4,F,R,52,3.0,2.0,30.0,1321,0.827,1.329,Nondemented


## Step 2: Handle Missing Values

In [6]:
print("Missing values before imputation:")
print(df.isnull().sum())


Missing values before imputation:
M/F       0
Hand      0
Age       0
Educ      0
SES      19
MMSE      0
eTIV      0
nWBV      0
ASF       0
Group     0
dtype: int64


In [7]:
# SES and Educ have some missing values among clinically-assessed subjects.
# MMSE has a small number of missing values too. These are legitimate cases worth
# imputing (unlike CDR/Group) since the subject WAS assessed -- a few individual
# fields are just incomplete. Use median imputation (robust to skew/outliers).

numeric_cols_with_na = ["Educ", "SES", "MMSE"]

imputer = SimpleImputer(strategy="median")
df[numeric_cols_with_na] = imputer.fit_transform(df[numeric_cols_with_na])

print("Missing values after imputation:")
print(df.isnull().sum())
print(f"\nShape after handling missing values: {df.shape}")


Missing values after imputation:
M/F      0
Hand     0
Age      0
Educ     0
SES      0
MMSE     0
eTIV     0
nWBV     0
ASF      0
Group    0
dtype: int64

Shape after handling missing values: (235, 10)


## Step 3: Encode Categorical Variables

In [8]:
print("Categorical columns and their values:")
print("M/F:", df["M/F"].unique())
print("Hand:", df["Hand"].unique())


Categorical columns and their values:
M/F: <StringArray>
['F', 'M']
Length: 2, dtype: str
Hand: <StringArray>
['R']
Length: 1, dtype: str


In [9]:
# M/F: binary -> encode as 0/1
df["M/F"] = df["M/F"].map({"M": 0, "F": 1})

# Hand: every subject in this dataset is right-handed (zero variance), so it
# carries no information for a classifier and is dropped rather than encoded.
if df["Hand"].nunique() <= 1:
    print(f"'Hand' has only {df['Hand'].nunique()} unique value -- dropping (zero variance, no predictive signal).")
    df = df.drop(columns=["Hand"])
else:
    le_hand = LabelEncoder()
    df["Hand"] = le_hand.fit_transform(df["Hand"])

print(f"\nShape after encoding categorical variables: {df.shape}")
df.head()


'Hand' has only 1 unique value -- dropping (zero variance, no predictive signal).

Shape after encoding categorical variables: (235, 9)


,M/F,Age,Educ,SES,MMSE,eTIV,nWBV,ASF,Group
0,1,74,2.0,3.0,29.0,1344,0.743,1.306,Nondemented
1,1,55,4.0,1.0,29.0,1147,0.810,1.531,Nondemented
2,1,73,4.0,3.0,27.0,1454,0.708,1.207,Demented
3,0,74,5.0,2.0,30.0,1636,0.689,1.073,Nondemented
4,1,52,3.0,2.0,30.0,1321,0.827,1.329,Nondemented


## Step 4: Encode Target Variable

In [10]:
# Binary target: Nondemented = 0, Demented = 1
target_map = {"Nondemented": 0, "Demented": 1}
df["Target"] = df["Group"].map(target_map)

print("Target mapping:", target_map)
print(df["Target"].value_counts())
print(f"\nShape after creating target variable: {df.shape}")


Target mapping: {'Nondemented': 0, 'Demented': 1}
Target
0    135
1    100
Name: count, dtype: int64

Shape after creating target variable: (235, 10)


In [11]:
# Separate features (X) and target (y); drop the now-redundant text label column
X = df.drop(columns=["Group", "Target"])
y = df["Target"]

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")


Feature matrix X shape: (235, 8)
Target vector y shape: (235,)

Feature columns: ['M/F', 'Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']


## Step 5: Train-Test Split (Stratified)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

print("\nClass balance check (proportion of Demented=1):")
print(f"  Train: {y_train.mean():.3f}")
print(f"  Test:  {y_test.mean():.3f}")
print(f"  Full:  {y.mean():.3f}")


X_train shape: (188, 8)
X_test shape:  (47, 8)
y_train shape: (188,)
y_test shape:  (47,)

Class balance check (proportion of Demented=1):
  Train: 0.426
  Test:  0.426
  Full:  0.426


**Note:** Stratification preserves the same Demented/Nondemented ratio in both train and test sets, which matters here given the class imbalance observed during EDA.

## Step 6: Apply StandardScaler

In [13]:
# Fit scaler on training data ONLY, then apply to both train and test
# (prevents data leakage from test set statistics into training)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep as DataFrames for readability / downstream use
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")
X_train_scaled.head()


X_train_scaled shape: (188, 8)
X_test_scaled shape:  (47, 8)


,M/F,Age,Educ,SES,MMSE,eTIV,nWBV,ASF
32,0.726949,-0.000878,-0.115639,-0.438622,0.802777,-0.023868,0.121285,-0.087830
127,0.726949,0.411739,-0.115639,-0.438622,0.527372,-0.781030,0.514058,0.768639
45,0.726949,-0.743589,-0.115639,-0.438622,0.802777,-0.402449,1.258260,0.321114
109,0.726949,0.164169,1.383679,-1.344785,0.802777,-0.886536,-0.126783,0.892094
163,0.726949,-0.083401,0.634020,-0.438622,-1.400465,0.205764,-1.201741,-0.311593


In [14]:
# Sanity check: scaled training features should have mean ~0, std ~1
print("Scaled X_train summary (should be ~0 mean, ~1 std):")
print(X_train_scaled.describe().loc[["mean", "std"]].round(3))


Scaled X_train summary (should be ~0 mean, ~1 std):
        M/F    Age   Educ    SES   MMSE   eTIV   nWBV    ASF
mean -0.000 -0.000 -0.000  0.000  0.000 -0.000  0.000 -0.000
std   1.003  1.003  1.003  1.003  1.003  1.003  1.003  1.003


## Step 7: Pipeline Summary — Shapes at Every Stage

The exact shape at every stage was printed live by the cells above as each transformation ran. Recap of the progression:
1. Raw load → all subjects, 12 raw columns
2. + `Group` derived from `CDR` → 13 columns
3. Drop `Unknown`-diagnosis subjects → fewer rows, 13 columns
4. Drop `ID`, `Delay`, `CDR` → 10 columns
5. Median-impute `Educ`, `SES`, `MMSE` → same shape, zero missing values
6. Encode `M/F`; drop `Hand` (zero variance) → 9 columns
7. Add `Target`, split into `X`/`y` → `X` has 8 feature columns
8. Stratified 80/20 train-test split → `X_train`/`X_test`, `y_train`/`y_test`
9. StandardScaler fit on train, applied to both → same shapes, standardized values


## Step 8: Pipeline Readiness Check for ML Models

In [15]:
print("=== Final ML-Ready Dataset Check ===\n")

print("1. No missing values in features:")
print(f"   X_train: {X_train_scaled.isnull().sum().sum()} missing")
print(f"   X_test:  {X_test_scaled.isnull().sum().sum()} missing")

print("\n2. All features are numeric:")
print(X_train_scaled.dtypes.value_counts())

print("\n3. Target is binary and properly encoded:")
print(f"   Unique values in y_train: {sorted(y_train.unique())}")
print(f"   Unique values in y_test:  {sorted(y_test.unique())}")

print("\n4. Final shapes:")
print(f"   X_train_scaled: {X_train_scaled.shape}")
print(f"   X_test_scaled:  {X_test_scaled.shape}")
print(f"   y_train:        {y_train.shape}")
print(f"   y_test:         {y_test.shape}")

print("\nPipeline complete. Data is ready for model training (e.g. Logistic Regression, SVM, Random Forest, etc.).")


=== Final ML-Ready Dataset Check ===

1. No missing values in features:
   X_train: 0 missing
   X_test:  0 missing

2. All features are numeric:
float64    8
Name: count, dtype: int64

3. Target is binary and properly encoded:
   Unique values in y_train: [np.int64(0), np.int64(1)]
   Unique values in y_test:  [np.int64(0), np.int64(1)]

4. Final shapes:
   X_train_scaled: (188, 8)
   X_test_scaled:  (47, 8)
   y_train:        (188,)
   y_test:         (47,)

Pipeline complete. Data is ready for model training (e.g. Logistic Regression, SVM, Random Forest, etc.).


**Final note:** `X_train_scaled`, `X_test_scaled`, `y_train`, and `y_test` are now ready to feed directly into any scikit-learn classifier. Given the class imbalance confirmed in Step 5, consider using `class_weight="balanced"` (for models that support it) or resampling techniques (e.g. SMOTE on the training set only) at the model-training stage.